# 02 — Test ComfyUI Connectivity

Goal: Verify local ComfyUI is reachable and can queue a simple workflow.

Prerequisites:
- ComfyUI running at `http://localhost:8188`
- A basic txt2img workflow JSON file ready (or use the built-in default)

In [ ]:
import os, json
import aiohttp

BASE_URL = os.getenv("COMFYUI_BASE_URL", "http://localhost:8188")
print(f"Target: {BASE_URL}")

## 2.1 Check server health

In [ ]:
async def check_server():
    try:
        async with aiohttp.ClientSession() as session:
            async with session.get(
                f"{BASE_URL}/system_stats",
                timeout=aiohttp.ClientTimeout(total=5.0)
            ) as resp:
                print(f"Status: {resp.status}")
                if resp.status == 200:
                    data = await resp.json()
                    print(f"System: {json.dumps(data, indent=2)[:500]}")
                else:
                    text = await resp.text()
                    print("Response:", text[:500])
    except Exception as e:
        print(f"ERROR: Cannot reach ComfyUI at {BASE_URL}")
        print(f"Details: {e}")
        print("\nTroubleshooting:")
        print("  1. Is ComfyUI running? (python main.py --listen)")
        print("  2. Is the port correct? Try: curl http://localhost:8188/system_stats")

await check_server()

## 2.2 Check available workflows (from workflows/ directory)

In [ ]:
import glob
wf_files = glob.glob("../workflows/*.json")
if wf_files:
    print("Available workflow files:")
    for f in wf_files:
        print(f"  - {f}")
else:
    print("No workflow JSON files found in backend/workflows/")
    print("You can create one by exporting a workflow from ComfyUI.")

## 2.3 Queue a simple prompt

This test sends a basic txt2img workflow to ComfyUI.
If no workflow file exists, it will be skipped.

In [ ]:
async def queue_prompt(session: aiohttp.ClientSession, workflow: dict) -> str | None:
    """Queue a workflow and return the prompt_id."""
    async with session.post(
        f"{BASE_URL}/prompt", json={"prompt": workflow}
    ) as resp:
        if resp.status == 200:
            data = await resp.json()
            print(f"Queued OK — prompt_id: {data.get('prompt_id')}")
            return data.get("prompt_id")
        else:
            text = await resp.text()
            print(f"Queue failed: {resp.status} — {text[:300]}")
            return None

async def get_history(session: aiohttp.ClientSession, prompt_id: str) -> dict | None:
    """Check workflow result."""
    async with session.get(f"{BASE_URL}/history/{prompt_id}") as resp:
        if resp.status == 200:
            data = await resp.json()
            return data.get(prompt_id)
        return None

async def run_workflow():
    # Use first available workflow or skip
    wf_files = glob.glob("../workflows/*.json")
    if not wf_files:
        print("Skipping workflow test — no workflow files found.")
        print("Add a workflow JSON to backend/workflows/ and re-run.")
        return

    with open(wf_files[0]) as f:
        workflow = json.load(f)
    print(f"Loaded workflow: {wf_files[0]}")

    async with aiohttp.ClientSession() as session:
        prompt_id = await queue_prompt(session, workflow)
        if prompt_id:
            import asyncio
            for i in range(30):
                await asyncio.sleep(1)
                result = await get_history(session, prompt_id)
                if result:
                    print(f"\nWorkflow completed!")
                    print(f"Outputs: {json.dumps(result.get('outputs', {}), indent=2)[:500]}")
                    break
            else:
                print("Workflow still running after 30s — check ComfyUI UI")

await run_workflow()

## 2.4 Test direct image generation via API

If you have a simple txt2img API workflow, test it here.

In [ ]:
# Placeholder for direct txt2img test
# Build a minimal workflow dict or use the ComfyUI API client
print("See the ComfyUI API docs for building a minimal prompt workflow.")

## Summary

| Check | Status |
|-------|--------|
| Server reachable | ⬜ |
| Workflow files exist | ⬜ |
| Can queue & execute workflow | ⬜ |

See the `backend/workflows/` directory for storing reusable ComfyUI workflow JSONs.